In [1]:
import sympy as sm
import math as m
import IPython.display as ipd


def f_series(f, h, n_terms=5):
    series = 0
    for n in range(n_terms):
        series += h**n / m.factorial(n) * f[n]
    return series

def f_sign(f, i=1):
    if i==0:
        return sm.latex(f)
    else:
        f_s = sm.latex(f) if str(f)[0]=='-' else '+' + sm.latex(f)
        return f_s.replace(' ', '') if i!=0 else sm.latex(f)

def f_latex(f_series, f, x, h):
    f_series_args = sorted(f_series.args, key=lambda t: sm.degree(t, h))
    fl = [sm.latex(fi) for fi in f]
    flsup = [fi.replace('_','^') + f'({x})' for fi in fl]
    f_args_latex = [f_sign(fi, i).replace(fl[i], '') + flsup[i] for i, fi in enumerate(f_series_args)]

    return "".join(f_args_latex)

def f_numeric(f, h=None, h_points=None, der=1, n_terms=3, fprint=False):
    fname = f.func.__name__
    fx = f.args[0]
    F = sm.Function(f'{fname}')
    f = sm.IndexedBase(f'{fname}')
    a = sm.IndexedBase("a")
    x = sm.symbols(f'{fx}')
    h = sm.symbols('h') if h is None else h
    h_points = [h, -h] if h_points is None else h_points

    a_vars = [a[i] for i in range(1, len(h_points)+1)]
    f_vars = [f[i] for i in range(0, n_terms)]
    fh_vars = [h**i * f[i] for i in range(0, n_terms)]
    Fs = [F(x + hi) for hi in h_points]

    eq_series = [f_series(f, hi, n_terms) for hi in h_points]

    bigOO = sm.Order(h**n_terms)
    order = sm.latex(bigOO)
    
    eq_sum = sum([(eq_series[i] - Fs[i])*a_vars[i] for i in range(len(h_points))])
    eq_org = sm.collect(eq_sum.expand(), fh_vars).subs(f[0], F(x))
    eq_coeffs = [eq_org.coeff(fh_vars[i]) if i!=der else eq_org.coeff(fh_vars[i]) - 1 for i in range(1, len(fh_vars))]
    a_sol = sm.solve(eq_coeffs, a_vars)

    eq_sol = sm.simplify(sm.solve(eq_org.subs(a_sol), f[der])[0])
    bigO = bigOO/sm.denom(eq_sol)
    
    if fprint:
        for hi, eqi in zip(h_points, eq_series):
            rhs = f_latex(eqi, f_vars, x, h)
            ipd.display(ipd.Markdown(fr'${f}({x} {f_sign(hi)}) \eqsim {rhs} + {order}$'))
        ipd.display(ipd.Markdown(fr'$\large \frac{{d^{{{der}}}}}{{d{x}^{{{der}}}}} {F}({x}) \eqsim {sm.latex(eq_sol)} + {sm.latex(bigO)}$'))

    return eq_sol, bigO

In [2]:
f = sm.Function('f')
x, h = sm.symbols('x, h')

print("\nnumerical derivatives:")
print("\nforward 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=2, h_points=[h], fprint=True)

print("\nbackward 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=2, h_points=[-h], fprint=True)

print("\nsymmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), fprint=True)

print("\nforward 2nd degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=2, n_terms=3, h_points=[h, 2*h], fprint=True)

print("\nbackward 2nd degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=2, n_terms=3, h_points=[-h, -2*h], fprint=True)

print("\nsymmetric 2nd degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=2, n_terms=4, fprint=True)


numerical derivatives:

forward 1st degree


$f(x +h) \eqsim {f}^{0}(x)+h{f}^{1}(x) + O\left(h^{2}\right)$

$\large \frac{d^{1}}{dx^{1}} f(x) \eqsim \frac{- f{\left(x \right)} + f{\left(h + x \right)}}{h} + O\left(h\right)$


backward 1st degree


$f(x -h) \eqsim {f}^{0}(x)-h{f}^{1}(x) + O\left(h^{2}\right)$

...

$\large \frac{d^{2}}{dx^{2}} f(x) \eqsim \frac{f{\left(x \right)} + f{\left(- 2 h + x \right)} - 2 f{\left(- h + x \right)}}{h^{2}} + O\left(h\right)$


symmetric 2nd degree


$f(x +h) \eqsim {f}^{0}(x)+h{f}^{1}(x)+\frac{h^{2}}{2}{f}^{2}(x)+\frac{h^{3}}{6}{f}^{3}(x) + O\left(h^{4}\right)$

$f(x -h) \eqsim {f}^{0}(x)-h{f}^{1}(x)+\frac{h^{2}}{2}{f}^{2}(x)-\frac{h^{3}}{6}{f}^{3}(x) + O\left(h^{4}\right)$

$\large \frac{d^{2}}{dx^{2}} f(x) \eqsim \frac{- 2 f{\left(x \right)} + f{\left(- h + x \right)} + f{\left(h + x \right)}}{h^{2}} + O\left(h^{2}\right)$

In [3]:
import numpy as np

f = sm.Function('f')
x, h = sm.symbols('x, h')

x0, h0 = 1, 0.043
fe = lambda x: np.exp(x)
df_exact = lambda x: fe(x)

print("\nforward 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=2, h_points=[h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\nsymmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=3, h_points=[h, -h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\n4-points symmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=5, h_points=[h, -h, 2*h, -2*h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\n6-points symmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=7, h_points=[h, -h, 2*h, -2*h, 3*h, -3*h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\n8-points symmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=9, h_points=[h, -h, 2*h, -2*h, 3*h, -3*h, 4*h, -4*h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\n10-points symmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=11, h_points=[h, -h, 2*h, -2*h, 3*h, -3*h, 4*h, -4*h, 5*h, -5*h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))

print("\n12-points symmetric 1st degree")
print("="*25)
f_num, f_O = f_numeric(f(x), der=1, n_terms=13, h_points=[h, -h, 2*h, -2*h, 3*h, -3*h, 4*h, -4*h, 5*h, -5*h, 6*h, -6*h], fprint=True)
df_num = lambda x, h: eval(sm.python(f_num).split("\n")[-1].split("=")[-1].replace('f','fe'))
print("numpy func df(x)/dx =", df_exact(x0))
print("numerical  df(x)/dx =", df_num(x0, h0))


forward 1st degree


$f(x +h) \eqsim {f}^{0}(x)+h{f}^{1}(x) + O\left(h^{2}\right)$

$\large \frac{d^{1}}{dx^{1}} f(x) \eqsim \frac{- f{\left(x \right)} + f{\left(h + x \right)}}{h} + O\left(h\right)$

numpy func df(x)/dx = 2.718281828459045
numerical  df(x)/dx = 2.7775716547247544

symmetric 1st degree


$f(x +h) \eqsim {f}^{0}(x)+h{f}^{1}(x)+\frac{h^{2}}{2}{f}^{2}(x) + O\left(h^{3}\right)$

...

$f(x -5h) \eqsim {f}^{0}(x)-5h{f}^{1}(x)+\frac{25h^{2}}{2}{f}^{2}(x)-\frac{125h^{3}}{6}{f}^{3}(x)+\frac{625h^{4}}{24}{f}^{4}(x)-\frac{625h^{5}}{24}{f}^{5}(x)+\frac{3125h^{6}}{144}{f}^{6}(x)-\frac{15625h^{7}}{1008}{f}^{7}(x)+\frac{78125h^{8}}{8064}{f}^{8}(x)-\frac{390625h^{9}}{72576}{f}^{9}(x)+\frac{390625h^{10}}{145152}{f}^{10}(x)-\frac{1953125h^{11}}{1596672}{f}^{11}(x)+\frac{9765625h^{12}}{19160064}{f}^{12}(x) + O\left(h^{13}\right)$

$f(x +6h) \eqsim {f}^{0}(x)+6h{f}^{1}(x)+18h^{2}{f}^{2}(x)+36h^{3}{f}^{3}(x)+54h^{4}{f}^{4}(x)+\frac{324h^{5}}{5}{f}^{5}(x)+\frac{324h^{6}}{5}{f}^{6}(x)+\frac{1944h^{7}}{35}{f}^{7}(x)+\frac{1458h^{8}}{35}{f}^{8}(x)+\frac{972h^{9}}{35}{f}^{9}(x)+\frac{2916h^{10}}{175}{f}^{10}(x)+\frac{17496h^{11}}{1925}{f}^{11}(x)+\frac{8748h^{12}}{1925}{f}^{12}(x) + O\left(h^{13}\right)$

$f(x -6h) \eqsim {f}^{0}(x)-6h{f}^{1}(x)+18h^{2}{f}^{2}(x)-36h^{3}{f}^{3}(x)+54h^{4}{f}^{4}(x)-\frac{324h^{5}}{5}{f}^{5}(x)+\frac{324h^{6}}{5}{f}^{6}(x)-\frac{1944h^{7}}{35}{f}^{7}(x)+\frac{1458h^{8}}{35}{f}^{8}(x)-\frac{972h^{9}}{35}{f}^{9}(x)+\frac{2916h^{10}}{175}{f}^{10}(x)-\frac{17496h^{11}}{1925}{f}^{11}(x)+\frac{8748h^{12}}{1925}{f}^{12}(x) + O\left(h^{13}\right)$

$\large \frac{d^{1}}{dx^{1}} f(x) \eqsim \frac{5 f{\left(- 6 h + x \right)} - 72 f{\left(- 5 h + x \right)} + 495 f{\left(- 4 h + x \right)} - 2200 f{\left(- 3 h + x \right)} + 7425 f{\left(- 2 h + x \right)} - 23760 f{\left(- h + x \right)} + 23760 f{\left(h + x \right)} - 7425 f{\left(2 h + x \right)} + 2200 f{\left(3 h + x \right)} - 495 f{\left(4 h + x \right)} + 72 f{\left(5 h + x \right)} - 5 f{\left(6 h + x \right)}}{27720 h} + O\left(h^{12}\right)$

numpy func df(x)/dx = 2.718281828459045
numerical  df(x)/dx = 2.718281828459046
